In [ ]:
from dataset import ImageDataset
from config import Config
from model import QualityModel, Trainer, get_data_loaders
import torch.optim as optim
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt
import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
from dataset import ImageDataset
from config import Config
import numpy as np
import torch
from train import classify_and_move_images
from model import QualityModel
import torch
from config import Config
from torchvision import transforms
from remove_empty import remove_empty_bad_quality_folders
import os
import shutil
import torch
from torch.utils.data import DataLoader
from dataset import ImageDataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from config import Config
from model import QualityModel

In [ ]:
!pip install torch
!pip install torchvision
!pip install matplotlib
!pip install numpy
!pip install scikit-learn



In [ ]:
# config.py
import torch

class Config:
    # Paths
    DATA_DIR = "/media/arnout/BS5/qc_nn_bsubt_kd"
 
    #C:/Users/Bart/qc_nn/training_data"
    GOOD_DIR = f"{DATA_DIR}/Good"
    BAD_DIR = f"{DATA_DIR}/bad"
    MODEL_PATH = f"{DATA_DIR}/quality_model_best_m4_v2.pth"
    TEST_GOOD = f"{DATA_DIR}/Test/Good_test"
    TEST_BAD = f"{DATA_DIR}/Test/Bad_test"
    #TEST_GOOD = f"{DATA_DIR}/test_good"
    #TEST_BAD = f"{DATA_DIR}/test_bad"
    # Model parameters
    IMAGE_SIZE = (224, 224)
    BATCH_SIZE = 8
    VAL_SPLIT = 0.2
    EPOCHS = 1000
    PATIENCE = 5
    LEARNING_RATE = 0.0001
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# dataset.py

!pip install opencv-python
!pip install tifffile

import os
import numpy as np
import torch
import cv2
import tifffile as tiff
from config import Config
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
class ImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, label=1):
        self.root_dir = root_dir
        self.transform = transform
        self.label = label
        self.images = [f for f in os.listdir(root_dir) if f.endswith("_C1.tiff")]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        filename = self.images[idx]
        img_path = os.path.join(self.root_dir, filename)
        try:
            image = tiff.imread(img_path)
            if len(image.shape) == 3:
                image = image[..., 0]
            image = image.astype(np.float32) / 65535.0  # Normalize 16-bit TIFF
            image = cv2.resize(image, Config.IMAGE_SIZE)
            image = np.expand_dims(image, axis=-1)
            if self.transform:
                image = self.transform(image)
            return image.to(torch.float32), torch.tensor([self.label], dtype=torch.float32), filename
        except Exception as e:
            print(f"(Error: {e})")
            # Return a dummy sample along with its filename so that the worker doesn't crash
            dummy = np.zeros((Config.IMAGE_SIZE[1], Config.IMAGE_SIZE[0], 1), dtype=np.float32)
            dummy = cv2.resize(dummy, Config.IMAGE_SIZE)
            dummy = np.expand_dims(dummy, axis=-1)
            if self.transform:
                dummy = self.transform(dummy)
            return dummy.to(torch.float32), torch.tensor([self.label], dtype=torch.float32), filename

In [ ]:
#model.py
!pip install tqdm

import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
from dataset import ImageDataset
from config import Config
import numpy as np
import torch

class QualityModel(nn.Module):
    def __init__(self):
        super(QualityModel, self).__init__()
        # Using EfficientNet instead of ResNet for better performance
        self.model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        # Modify first conv layer for grayscale input
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        # Modify classifier for binary output
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, 1)

    def forward(self, x):
        return torch.sigmoid(self.model(x))

# data_utils.py
def get_data_loaders():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.ConvertImageDtype(torch.float32),
    ])

    good_dataset = ImageDataset(root_dir=Config.GOOD_DIR, transform=transform, label=1)
    bad_dataset = ImageDataset(root_dir=Config.BAD_DIR, transform=transform, label=0)

    # Split datasets
    good_train_size = int((1 - Config.VAL_SPLIT) * len(good_dataset))
    good_val_size = len(good_dataset) - good_train_size
    bad_train_size = int((1 - Config.VAL_SPLIT) * len(bad_dataset))
    bad_val_size = len(bad_dataset) - bad_train_size

    good_train, good_val = random_split(good_dataset, [good_train_size, good_val_size])
    bad_train, bad_val = random_split(bad_dataset, [bad_train_size, bad_val_size])

    train_dataset = ConcatDataset([good_train, bad_train])
    val_dataset = ConcatDataset([good_val, bad_val])

    # Create balanced sampler
    train_labels = np.array([int(sample[1].item()) for sample in train_dataset])
    class_counts = np.bincount(train_labels)
    class_weights = 1. / class_counts
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=1)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=1)

    return train_loader, val_loader

# trainer.py
from tqdm import tqdm

class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, scheduler, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.best_loss = float('inf')
        self.early_stop_count = 0

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        progress_bar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        
        # Unpack three values (image, label, filename) and ignore the filename
        for inputs, labels, _ in progress_bar:
            inputs, labels = inputs.to(self.device), labels.to(self.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())
            
        return total_loss / len(self.train_loader)

    def validate(self):
        self.model.eval()
        total_loss = 0
        with torch.no_grad():
            for inputs, labels, _ in self.val_loader:  # Unpack and ignore the filename
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
        
        return total_loss / len(self.val_loader)

In [ ]:
#train
from dataset import ImageDataset
from config import Config
from model import QualityModel, Trainer, get_data_loaders
import torch.optim as optim
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt


def main():
    train_losses = []
    val_losses = []
    # Get data loaders
    train_loader, val_loader = get_data_loaders()
    print(f"✅ Training data: {len(train_loader.dataset)} images, "
          f"Validation data: {len(val_loader.dataset)} images")

    # Initialize model and training components
    model = QualityModel().to(Config.DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
    #scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5, verbose=True)
    # Initialize trainer
    trainer = Trainer(model, train_loader, val_loader, criterion, optimizer, scheduler, Config.DEVICE)
    
    for epoch in range(Config.EPOCHS):
        train_loss = trainer.train_epoch(epoch)
        val_loss = trainer.validate()
        
        # Store losses
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        print(f"Epoch {epoch+1}/{Config.EPOCHS}")
        print(f"Training Loss: {train_loss:.4f}")
        print(f"Validation Loss: {val_loss:.4f}")
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < trainer.best_loss:
            trainer.best_loss = val_loss
            trainer.early_stop_count = 0
            torch.save(model.state_dict(), Config.MODEL_PATH)
            print("✅ Model improved & saved!")
        else:
            trainer.early_stop_count += 1
            print(f"Early stopping counter: {trainer.early_stop_count}/{Config.PATIENCE}")
            
        if trainer.early_stop_count >= Config.PATIENCE:
            print("⏹️ Early stopping triggered!")
            break
    
    # Visualization of loss curves after training
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()
    
    print("🎉 Training complete!")

if __name__ == "__main__":
    main()

# inference.py
import shutil
from torch.utils.data import DataLoader
def classify_and_move_images(root_folder, model, transform):
    model.eval()
    
    # Get subfolders once
    subfolders = [f for f in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, f))]

    for subfolder in subfolders:
        subfolder_path = os.path.join(root_folder, subfolder)
        bad_quality_folder = f"{subfolder_path}_bad_Q"
        os.makedirs(bad_quality_folder, exist_ok=True)

        # Create dataset and DataLoader (minimal memory footprint with batch_size=1)
        dataset = ImageDataset(subfolder_path, transform=transform)
        dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)  # num_workers=0 to avoid memory overhead

        for img_tensor, _, filename in dataloader:
            # Only evaluate images ending with _C1.tiff
            if not filename[0].endswith("_C1.tiff"):
                continue

            img_tensor = img_tensor.to(Config.DEVICE)
            with torch.no_grad():
                prediction = model(img_tensor).item()

            # Explicitly delete tensor to free memory
            del img_tensor
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Move files if low quality
            if prediction < 0.5:
                base = filename[0].replace("_C1.tiff", "")
                for suffix in ["_C1.tiff", "_C2.tiff", "_C3.tiff", "_C4.tiff", "_C5.tiff"]:
                    target_filename = base + suffix
                    source_path = os.path.join(subfolder_path, target_filename)
                    if os.path.exists(source_path):
                        shutil.move(source_path, os.path.join(bad_quality_folder, target_filename))
                        print(f"Moved {target_filename} to {bad_quality_folder}")

        print(f"Processed folder: {subfolder_path}")

    # Final cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
#evaluate model
import os
import shutil
import torch
from torch.utils.data import DataLoader
from dataset import ImageDataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from config import Config
from model import QualityModel


DATA_DIR = "/media/arnout/BS5/qc_nn_bsubt_kd"
TEST_GOOD = f"{DATA_DIR}/Test/Good_test"
TEST_BAD = f"{DATA_DIR}/Test/Bad_test"

# Load the model
model = QualityModel().to(Config.DEVICE)
model.load_state_dict(torch.load(Config.MODEL_PATH, weights_only=True))
model.eval()

# Define the transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float32),
])

# Load the test datasets
test_good_dataset = ImageDataset(TEST_GOOD, transform=transform, label=1)
test_bad_dataset = ImageDataset(TEST_BAD, transform=transform, label=0)

# Create data loaders
test_good_loader = DataLoader(test_good_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
test_bad_loader = DataLoader(test_bad_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

# Function to evaluate the model
def evaluate_model(loader, label):
    all_preds = []
    all_labels = []
    wrong_preds = []
    with torch.no_grad():
        for images, _, filenames in loader:
            for img_tensor, filename in zip(images, filenames):
                # Only evaluate images ending with _C1.tiff
                if not filename.endswith("_C1.tiff"):
                    continue

                img_tensor = img_tensor.to(Config.DEVICE)
                prediction = model(img_tensor.unsqueeze(0)).item()

                # Convert prediction to binary class
                pred_label = 1 if prediction >= 0.5 else 0
                all_preds.append(pred_label)
                all_labels.append(label)

                # Check if the prediction is wrong
                if pred_label != label:
                    wrong_preds.append(filename)
    return all_preds, all_labels, wrong_preds

# Evaluate on good and bad test sets
good_preds, good_labels, good_wrong_preds = evaluate_model(test_good_loader, 1)
bad_preds, bad_labels, bad_wrong_preds = evaluate_model(test_bad_loader, 0)

# Combine results
all_preds = good_preds + bad_preds
all_labels = good_labels + bad_labels
wrong_preds = good_wrong_preds + bad_wrong_preds

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

# Print metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Print wrongfully predicted samples
print("Wrongfully predicted samples:")
for filename in wrong_preds:
    print(filename)

In [ ]:
#inference
from train import classify_and_move_images
from model import QualityModel
import torch
from config import Config
from torchvision import transforms
from remove_empty import remove_empty_bad_quality_folders

#base_path = 'I:/Export Bart/Stationary_phase_screen'
#plate_ids = ['89B', 'AA', 'AB', 'B']  # Add more plate IDs as needed
base_path = '/media/arnout/BS5/ScreenCrisprI_Arnout/Bstubt_KD_screen'
plate_ids = ['1_T0','1_T1','1_T2','2_T0','2_T2','3_T0','3_T1','3_T2','4_T0','4_T1','4_T2','5_T0_T1','5_T2' ]  # Add more plate IDs as needed
ev_dirs = [f"{base_path}/PLATE{plate_id}" for plate_id in plate_ids]

# Create transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float32),
])

# Load model
model = QualityModel().to(Config.DEVICE)
print('Device:', Config.DEVICE)
model.load_state_dict(torch.load(Config.MODEL_PATH))

# Loop through each directory
for ev_dir in ev_dirs:
    print(f"Processing directory: {ev_dir}")
    classify_and_move_images(ev_dir, model, transform)
    remove_empty_bad_quality_folders(ev_dir)
    print(f"Finished processing: {ev_dir}")

In [ ]:
import os
import shutil

def move_back_files(root_folder):
    """
    Moves files from directories ending with '_bad_Q' back to their original folders.
    The original folder is assumed to be the same as the '_bad_Q' folder with that suffix removed.
    """
    # List all directories in root_folder ending with "_bad_Q"
    for folder_name in os.listdir(root_folder):
        folder_path = os.path.join(root_folder, folder_name)
        if os.path.isdir(folder_path) and folder_name.endswith("_bad_Q"):
            # Determine the original folder name by removing the "_bad_Q" suffix
            original_folder_name = folder_name[:-6]  # Remove last 6 characters ("_bad_Q")
            original_folder_path = os.path.join(root_folder, original_folder_name)

            if not os.path.exists(original_folder_path):
                print(f"Original folder '{original_folder_path}' does not exist. Skipping '{folder_path}'.")
                continue

            # Move all files from the bad folder back to the original folder
            for file in os.listdir(folder_path):
                source_path = os.path.join(folder_path, file)
                target_path = os.path.join(original_folder_path, file)
                try:
                    shutil.move(source_path, target_path)
                    print(f"Moved '{file}' from '{folder_path}' back to '{original_folder_path}'.")
                except Exception as e:
                    print(f"Error moving '{file}': {e}")

            # Optionally, remove the empty bad folder
            try:
                os.rmdir(folder_path)
                print(f"Removed empty folder '{folder_path}'.")
            except Exception as e:
                print(f"Could not remove folder '{folder_path}': {e}")

if __name__ == "__main__":
    base_path = '/media/arnout/BS5/ScreenCrisprI_Arnout/Bstubt_KD_screen'
    plate_ids = ['1_T0','1_T1','1_T2','2_T0','2_T2','3_T0','3_T1','3_T2','4_T0','4_T1','4_T2','5_T0_T1','5_T2' ]
     # Add more plate IDs as needed

    ev_dirs = [f"{base_path}/PLATE{plate_id}" for plate_id in plate_ids]
    for ev_dir in ev_dirs:
        print(f"Processing directory: {ev_dir}")
        move_back_files(ev_dir)
        print(f"Finished processing: {ev_dir}")
    

In [ ]:
import os
import torch
from torchvision import transforms
from train import classify_and_move_images
from model import QualityModel
from config import Config
from remove_empty import remove_empty_bad_quality_folders

# 1. Update the base path to point to your Scratch via the link
# This uses the 'view_scratch' link you created in your home directory
base_path = os.path.expanduser('~/view_scratch') 

# 2. Update to your specific Plate ID
plate_ids = ['2_T1'] # Combined with 'PLATE' below, this looks for PLATE2_T1

# Construct the list of directories to process
ev_dirs = [f"{base_path}/PLATE{plate_id}" for plate_id in plate_ids]

# Create transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float32),
])

# Load model
model = QualityModel().to(Config.DEVICE)
print('Device:', Config.DEVICE)
model.load_state_dict(torch.load(Config.MODEL_PATH))

# Loop through each directory
for ev_dir in ev_dirs:
    if os.path.exists(ev_dir):
        print(f"Processing directory: {ev_dir}")
        classify_and_move_images(ev_dir, model, transform)
        remove_empty_bad_quality_folders(ev_dir)
        print(f"Finished processing: {ev_dir}")
    else:
        print(f"Error: Directory not found at {ev_dir}")